In [1]:

import pandas as pd
from pandas import DataFrame

In [8]:
urbanisation = pd.read_csv('urbanisation_natalite.csv',sep=',', header=0, index_col=0)

urbanisation.head()

,pays,taux_urbanisation,taux_natalite
Observations,,,
1,Canada,55.0,16.2
2,Costa-Rica,27.3,30.5
3,Cuba,33.3,16.9
4,USA,56.5,16.0
5,El Salvador,11.5,40.2


In [6]:
numData = urbanisation.select_dtypes(include='number')

numData

,taux_urbanisation,taux_natalite
Observations,,
1,55.0,16.2
2,27.3,30.5
3,33.3,16.9
4,56.5,16.0
5,11.5,40.2
6,14.2,38.4
7,13.9,41.3
8,19.0,43.9
9,33.1,28.3


In [26]:
statistique = pd.DataFrame(numData[['taux_urbanisation', 'taux_natalite']].mean(), columns=['moyenne'])

In [24]:
moyenne
#les moyennes des deux sont assez proches avec moins de 2 points d'écart

,moyenne
taux_urbanisation,29.792857
taux_natalite,31.107143


In [27]:
statistique['medianne'] = numData[['taux_urbanisation', 'taux_natalite']].median()

In [28]:
statistique
#les médianes des deux sont assez proches avec moins de 2 points d'écart
#sur les lignes respectives, on a peu d'écart, cela suggère une distribution homogène des données


,moyenne,medianne
taux_urbanisation,29.792857,30.8
taux_natalite,31.107143,31.8


In [30]:
statistique['mean_med_diff']=statistique['moyenne']-statistique['medianne']
statistique

,moyenne,medianne,mean_med_diff
taux_urbanisation,29.792857,30.8,-1.007143
taux_natalite,31.107143,31.8,-0.692857


In [33]:
statistique[['q1','q2','q3']] = numData.quantile(q=[0.25,0.5,0.75]).transpose()

statistique

#COMPARAISON Q2 ET Q3
#l'écart entre q2 et q3 est très faible dans les deux cas (7 et 8 points). Elles sont donc très concentré vers le centre dans la partie supérieur

#COMPARAISON MOYENNE Q3
#l'écart entre la moyenne et q3 est aussi faible (7 et 8,5 points). On peut encore supposer que les valeurs sont très concentré vers le centre avec peu d'extrème côté supérieur.Cela confirme l'analyse précédente.

,moyenne,medianne,mean_med_diff,q1,q2,q3
taux_urbanisation,29.792857,30.8,-1.007143,15.40,30.8,37.55
taux_natalite,31.107143,31.8,-0.692857,25.45,31.8,39.75


In [34]:
statistique['min'] = numData[['taux_urbanisation', 'taux_natalite']].min()
statistique['max'] = numData[['taux_urbanisation', 'taux_natalite']].max()
statistique['min_max_diff'] = statistique['max'] - statistique['min']


#min max diff est plus élevé sur la première ligne qui a un plus grand écart entre q1 et q3 que sur la seconde ligne. Comme mean med diff sont quasi au centre, on peut penser que plus min max diff sera élevé plus les valeurs seront dispersé (et à l'inverse plus bas = plus proche du centre)
statistique

,moyenne,medianne,mean_med_diff,q1,q2,q3,min,max,min_max_diff
taux_urbanisation,29.792857,30.8,-1.007143,15.40,30.8,37.55,6.8,56.5,49.7
taux_natalite,31.107143,31.8,-0.692857,25.45,31.8,39.75,16.0,44.2,28.2


In [37]:
numData.std()

taux_urbanisation    15.568630
taux_natalite         9.996727
dtype: float64

In [47]:
borne = [
    statistique.loc['taux_urbanisation','moyenne'] - numData['taux_urbanisation'].std(),
    statistique.loc['taux_urbanisation','moyenne'] + numData['taux_urbanisation'].std(),
 ]
borne

[np.float64(14.224226735937826), np.float64(45.36148754977646)]

In [50]:
numData['Urbanisation_in']=numData['taux_urbanisation'].between(borne[0],borne[1]).astype(int)

numData

,taux_urbanisation,taux_natalite,Urbanisation_in
Observations,,,
1,55.0,16.2,0
2,27.3,30.5,1
3,33.3,16.9,1
4,56.5,16.0,0
5,11.5,40.2,0
6,14.2,38.4,0
7,13.9,41.3,0
8,19.0,43.9,1
9,33.1,28.3,1


In [54]:
dataOutliers= pd.DataFrame(numData[numData['Urbanisation_in']==0])

dataOutliers

,taux_urbanisation,taux_natalite,Urbanisation_in
Observations,,,
1,55.0,16.2,0
4,56.5,16.0,0
5,11.5,40.2,0
6,14.2,38.4,0
7,13.9,41.3,0
12,6.8,24.6,0


In [60]:
dataOutliers['taux_urbanisation'].mean()

np.float64(26.316666666666666)

In [58]:
dataOutliers['taux_urbanisation'].median()

np.float64(14.05)

In [62]:
dataOutliers['taux_urbanisation'].quantile(q=[0.25,0.5,0.75])

# q1 q2 q3 sur les outliers montrent un écart de dispersion. Le quart des données entre q1 et q2 (12 et 14) est ultra-concentré juste sous le seuil. À l'inverse, le quart suivant entre q2 et q3 (14 et 44) s'étale sur 30 points, prouvant qu'une minorité d'outliers s'envole vers des niveaux élevés.

0.25    12.10
0.50    14.05
0.75    44.80
Name: taux_urbanisation, dtype: float64